In [ ]:
import numpy as np
import math
import random
import csv
import pandas as pd
import scipy
import scipy.optimize as opt
from bayes_opt import BayesianOptimization
from bayes_opt.logger import JSONLogger
from bayes_opt.event import Events
from bayes_opt.util import load_logs

In [ ]:
def rl_ai_subject(gamma,lr):
    trial_num = 120
    if_can_ask = IF_CAN_ASK_ACTIVE_INFERENCE[subject]
    action_stay_cue = ACTION_STAY_CUE_ACTIVE_INFERENCE[subject]
    result_stay_cue = RESULT_STAY_CUE_ACTIVE_INFERENCE[subject]
    action_safe_risk = ACTION_SAFE_RISK_ACTIVE_INFERENCE[subject]
    result_safe_risk = RESULT_SAFE_RISK_ACTIVE_INFERENCE[subject]
    log_p = 0
    value = np.array([[6,6,6],[6,6,6]])
    for i in range(120):
        if if_can_ask[i] == 1:
            log_p += np.log(P_action_softmax(value,gamma,[1,0,0],action_stay_cue[i]))
        if result_stay_cue[i]==0:
            state = [0,0.5,0.5]
            log_p += np.log(P_action_con0(value,gamma)[action_safe_risk[i]])
        elif result_stay_cue[i]==1:
            state = [0,1,0]
            log_p += np.log(P_action_con1(value,gamma)[action_safe_risk[i]])
        elif result_stay_cue[i]==2:
            state=[0,0,1]
            log_p += np.log(P_action_con2(value,gamma)[action_safe_risk[i]])
        else:
            print('ERROR')
        value = learn(value,lr,state,action_safe_risk[i],result_safe_risk[i])
    return log_p

pbounds_rl = {'gamma':(0.001,10),'lr':(0.0,1)}
optimizer_rl = BayesianOptimization(
    f=rl_ai_subject,
    pbounds=pbounds_rl,
    random_state=1,allow_duplicate_points=True)

LL_rl_ai = []

for i in range(25):
    load_logs(optimizer_rl, logs=["./model_recovery/mr_logs_rl_ai_"+str(i+1)+".log.json"])
    LL_rl_ai.append(optimizer.max['target'])